[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/09.%20The%20Quay%20Crane%20Scheduling%20Problem/P9-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 9. The Quay Crane Scheduling Problem
## Tier 6 — Autonomous & Self-Optimizing Ecosystem (Multi-Agent System)

Tier 5 gave us a centralized Digital Twin. Tier 6 decentralizes control: each crane, vessel, yard block, and the terminal controller becomes an **autonomous agent** that negotiates with others via auction-based protocols. This mimics biological ecosystems where local interactions create global efficiency. The system converges to a **Nash equilibrium** where no single agent can improve its payoff by unilaterally changing its strategy.

### Learning goals
- Understand how to model terminal components as **autonomous agents** with local objectives.
- Learn **auction-based task allocation** and **game-theoretic payoff functions**.
- See how **decentralized coordination** emerges from local negotiations.
- Compare resilience of the multi-agent system vs centralized optimization under disruptions.

### What this notebook outputs
- A 3-vessel, 12-crane, 8-yard-block multi-agent simulation.
- **Auction logs** showing bids, winners, and task allocations.
- **Payoff evolution** demonstrating convergence toward equilibrium.
- A **disruption scenario** (crane failure) with automatic task redistribution.
- Performance comparison: multi-agent vs centralized (baseline).

### Why the instance is decentralized
We use multiple agents to highlight how local decisions (crane bids, vessel contracts) scale to system-wide optimization without a central planner.

In [1]:
from itertools import product
from typing import Tuple, List, Dict, Set

# Environment check (no installs here)
#
# Best practice: dependencies are preinstalled in the Docker/JupyterHub image.
# If you are running locally, install packages once in your environment.

try:
    import random
    from dataclasses import dataclass, field
    from typing import List, Dict, Tuple, Any, Optional
    from collections import defaultdict

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

Dependencies imported successfully.


## Multi-agent architecture overview

Our ecosystem includes four agent types:

1. **Crane Agents**: Maximize throughput while minimizing energy and coordination penalties.
2. **Vessel Agents**: Represent shipping lines; negotiate service priority based on cargo value and urgency.
3. **Yard Block Agents**: Manage storage and internal transport; negotiate delivery schedules with quay cranes.
4. **Terminal Control Agent**: Enforces safety constraints and collects global metrics (does not dictate schedules).

### Auction-based task allocation
- Vessel agents announce **container batches** (tasks) with required processing time and value.
- Crane agents submit **bids** based on their current capacity, position, and energy cost.
- Yard agents bid for **delivery slots** to receive containers from cranes.
- Winners are selected by **lowest cost-performance ratio** (bid / task value).

In [ ]:
# ----------------------------
# Agent data models
# ----------------------------
@dataclass(frozen=True)
class Task:
    id: str
    vessel_id: str
    teu: int  # TEU to process
    processing_time_min: float  # minutes
    value: float  # monetary value (used for vessel payoff)
    deadline: Optional[float] = None  # optional deadline (minutes from start)


@dataclass(frozen=True)
class Bid:
    agent_id: str
    task_id: str
    cost: float  # bid cost (lower is better)
    capacity: float  # agent capacity (e.g., TEU/hour)
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CraneAgent:
    id: str
    position: int  # bay position (0..20)
    productivity_base: float  # TEU/hour
    energy_cost_per_hour: float
    current_load: float = 0.0  # current TEU assigned
    total_capacity: float = 0.0

    def effective_capacity(self) -> float:
        # Capacity reduces with load (interference)
        return self.productivity_base * (1 - 0.05 * (self.current_load / 1000))

    def submit_bid(self, task: Task) -> Bid:
        # Compute bid cost based on distance, energy, and current load
        distance = abs(self.position - int(task.id.split('-')[-1]))  # assume task id includes bay
        travel_cost = distance * 0.5  # $ per bay distance
        processing_cost = task.processing_time_min / 60 * self.energy_cost_per_hour
        load_penalty = self.current_load * 0.01
        total_cost = travel_cost + processing_cost + load_penalty
        return Bid(
            agent_id=self.id,
            task_id=task.id,
            cost=total_cost,
            capacity=self.effective_capacity(),
            metadata={"position": self.position, "load": self.current_load},
        )


@dataclass
class VesselAgent:
    id: str
    total_teus: int
    tasks: List[Task] = field(default_factory=list)
    utility_weights: Dict[str, float] = field(default_factory=lambda: {"throughput": 0.5, "delay": 0.3, "cost": 0.2})

    def announce_tasks(self) -> List[Task]:
        return self.tasks

    def compute_utility(self, completed_tasks: List[Task], total_cost: float) -> float:
        total_teus = sum(t.teu for t in completed_tasks)
        total_value = sum(t.value for t in completed_tasks)
        # Simple utility: weighted sum of throughput, value, and cost
        utility = (
            self.utility_weights["throughput"] * total_teus
            + self.utility_weights["delay"] * total_value
            - self.utility_weights["cost"] * total_cost
        )
        return utility


@dataclass
class YardBlockAgent:
    id: str
    capacity: int  # TEU
    current_utilization: float = 0.0
    transport_capacity: float = 40.0  # TEU/hour internal transport

    def submit_bid(self, task: Task) -> Bid:
        # Bid based on available space and transport capacity
        if self.capacity - self.current_utilization < task.teu:
            # Not enough space; submit high cost
            return Bid(agent_id=self.id, task_id=task.id, cost=1e6, capacity=0.0)
        # Cost based on utilization (higher utilization = higher cost)
        utilization_cost = self.current_utilization / self.capacity * 10
        return Bid(
            agent_id=self.id,
            task_id=task.id,
            cost=utilization_cost,
            capacity=self.transport_capacity,
            metadata={"utilization": self.current_utilization},
        )


@dataclass
class TerminalControlAgent:
    id: str = "TerminalControl"
    safety_constraints: Dict[str, Any] = field(default_factory=lambda: {"min_crane_distance": 2})

    def validate_allocation(self, allocation: Dict[str, List[str]]) -> bool:
        # Simplified safety check: ensure no two cranes assigned to tasks too close
        # In a full implementation, this would check positions and times
        return True


# ----------------------------
# Scenario setup (3 vessels, 12 cranes, 8 yard blocks)
# ----------------------------
def create_scenario(seed: int = 42) -> Tuple[List[VesselAgent], List[CraneAgent], List[YardBlockAgent], TerminalControlAgent]:
    random.seed(seed)
    np.random.seed(seed)
    # Vessels
    vessels = [
        VesselAgent(id="V1", total_teus=2200),
        VesselAgent(id="V2", total_teus=1800),
        VesselAgent(id="V3", total_teus=2500),
    ]
    # Create tasks for each vessel (simplified: 1 task per vessel)
    for i, v in enumerate(vessels):
        task_id = f"{v.id}-Task-{i+1}"
        task = Task(
            id=task_id,
            vessel_id=v.id,
            teu=v.total_teus,
            processing_time_min=v.total_teus / 25.0,  # assume 25 TEU/hr
            value=v.total_teus * 10,  # $10 per TEU
        )
        v.tasks.append(task)
    # Cranes
    cranes = [
        CraneAgent(
            id=f"C{i}",
            position=random.randint(0, 20),
            productivity_base=25 + random.uniform(-2, 2),
            energy_cost_per_hour=50 + random.uniform(-5, 5),
        )
        for i in range(12)
    ]
    # Yard blocks
    yards = [
        YardBlockAgent(id=f"Y{i}", capacity=500 + random.randint(-50, 50))
        for i in range(8)
    ]
    control = TerminalControlAgent()
    return vessels, cranes, yards, control


vessels, cranes, yards, control = create_scenario(seed=42)
print(f"Created {len(vessels)} vessels, {len(cranes)} cranes, {len(yards)} yard blocks.")

Generation  40: Best=2913.09, Avg=2937.27, Diversity=1.5
Generation  60: Best=2913.09, Avg=2933.07, Diversity=0.6
Iteration    0: Temp=500.00, Current=3732.90, Best=3732.90, Accept=1.000
Iteration  100: Temp=183.02, Current=3388.42, Best=3022.62, Accept=0.827
Iteration  200: Temp=66.99, Current=3358.33, Best=3022.62, Accept=0.689
Iteration  300: Temp=26.97, Current=3109.08, Best=3022.62, Accept=0.481
Iteration  400: Temp=9.87, Current=3042.30, Best=3022.62, Accept=0.410
Iteration  500: Temp=3.61, Current=2993.49, Best=2992.02, Accept=0.200
Iteration  600: Temp=1.32, Current=2959.03, Best=2959.03, Accept=0.220
  GA: 2.8s, fitness=2873
  SA: 0.2s, fitness=2959

🧪 Testing 35 ships, 8 berths...
Created 3 vessels, 12 cranes, 8 yard blocks.


def run_auction(
    vessels: List[VesselAgent],
    cranes: List[CraneAgent],
    yards: List[YardBlockAgent],
    control: TerminalControlAgent,
    log: bool = True,
) -> Dict[str, Any]:
    """Run a sealed-bid auction for all tasks."""
    auction_log = []
    allocation = {"crane": {}, "yard": {}}
    # Step 1: collect all tasks
    all_tasks = []
    for v in vessels:
        all_tasks.extend(v.announce_tasks())
    # Step 2: collect bids for each task
    for task in all_tasks:
        crane_bids = [c.submit_bid(task) for c in cranes]
        yard_bids = [y.submit_bid(task) for y in yards]
        # Step 3: select crane winner (lowest cost per TEU)
        best_crane_bid = min(crane_bids, key=lambda b: b.cost / task.teu if b.capacity > 0 else float("inf"))
        best_yard_bid = min(yard_bids, key=lambda b: b.cost if b.capacity > 0 else float("inf"))
        # Record
        auction_log.append(
            {
                "task_id": task.id,
                "crane_winner": best_crane_bid.agent_id if best_crane_bid.capacity > 0 else None,
                "crane_cost": best_crane_bid.cost if best_crane_bid.capacity > 0 else 0,
                "yard_winner": best_yard_bid.agent_id if best_yard_bid.capacity > 0 else None,
                "yard_cost": best_yard_bid.cost if best_yard_bid.capacity > 0 else 0,
            }
        )
        # Update allocation
        if best_crane_bid.capacity > 0:
            allocation["crane"][task.id] = best_crane_bid.agent_id
            # Update crane load
            crane = next(c for c in cranes if c.id == best_crane_bid.agent_id)
            crane.current_load += task.teu
        if best_yard_bid.capacity > 0:
            allocation["yard"][task.id] = best_yard_bid.agent_id
            # Update yard utilization
            yard = next(y for y in yards if y.id == best_yard_bid.agent_id)
            yard.current_utilization += task.teu
    # Validate allocation
    valid = control.validate_allocation(allocation["crane"])
    result = {
        "allocation": allocation,
        "auction_log": pd.DataFrame(auction_log),
        "valid": valid,
    }
    if log:
        print(f"Auction completed. Valid: {valid}")
        print(result["auction_log"])
    return result


auction_result = run_auction(vessels, cranes, yards, control, log=True)

In [ ]:
try:
    try:
        ## Step 2 — Payoff computation and Nash equilibrium check
        
        We compute each agent's payoff based on the allocation:
        - **Crane payoff**: α·productivity - β·energy_cost - γ·coordination_penalty
        - **Vessel payoff**: utility from completed tasks minus payments to cranes/yards.
        - **Yard payoff**: revenue from storage minus transport cost.
        
        We then check if any agent can improve its payoff by unilaterally changing its bid (simplified Nash check).
    except Exception as e:
        print(f'Error in cell: {e}')
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unterminated string literal (detected at line 4) (<string>, line 4)...]

def compute_payoffs(
    vessels: List[VesselAgent],
    cranes: List[CraneAgent],
    yards: List[YardBlockAgent],
    allocation: Dict[str, Dict[str, str]],
    auction_log: pd.DataFrame,
) -> Dict[str, float]:
    """Compute payoffs for all agents given the allocation."""
    payoffs = {}
    # Crane payoffs
    alpha, beta, gamma = 0.5, 0.3, 0.2
    for crane in cranes:
        # Productivity (TEU processed)
        tasks_assigned = [t for t, cid in allocation["crane"].items() if cid == crane.id]
        teu_processed = 0
        for task_id in tasks_assigned:
            for vessel in vessels:
                for task in vessel.tasks:
                    if task.id == task_id:
                        teu_processed += task.teu
                        break
        productivity = teu_processed
        # Energy cost (based on processing time)
        total_processing_time = 0
        for task_id in tasks_assigned:
            for vessel in vessels:
                for task in vessel.tasks:
                    if task.id == task_id:
                        total_processing_time += task.processing_time_min
                        break
        energy_cost = total_processing_time / 60 * crane.energy_cost_per_hour
        # Coordination penalty (simplified: based on load variance)
        coordination_penalty = crane.current_load * 0.01
        payoff = alpha * productivity - beta * energy_cost - gamma * coordination_penalty
        payoffs[crane.id] = payoff
    # Vessel payoffs
    for vessel in vessels:
        completed_tasks = [t for t in vessel.tasks if t.id in allocation["crane"]]
        total_cost = 0
        for t in completed_tasks:
            crane_row = auction_log.loc[auction_log["task_id"] == t.id]
            if not crane_row.empty:
                crane_cost = crane_row["crane_cost"].iloc[0] if pd.notna(crane_row["crane_cost"].iloc[0]) else 0
                yard_cost = crane_row["yard_cost"].iloc[0] if pd.notna(crane_row["yard_cost"].iloc[0]) else 0
                total_cost += crane_cost + yard_cost
        utility = vessel.compute_utility(completed_tasks, total_cost)
        payoffs[vessel.id] = utility
    # Yard payoffs
    for yard in yards:
        tasks_assigned = [t for t, yid in allocation["yard"].items() if yid == yard.id]
        teu_received = 0
        for task_id in tasks_assigned:
            for vessel in vessels:
                for task in vessel.tasks:
                    if task.id == task_id:
                        teu_received += task.teu
                        break
        revenue = teu_received * 2  # $2 per TEU storage revenue
        transport_cost = teu_received / yard.transport_capacity * 10  # $10 per TEU transport
        payoff = revenue - transport_cost
        payoffs[yard.id] = payoff
    return payoffs


payoffs = compute_payoffs(vessels, cranes, yards, auction_result["allocation"], auction_result["auction_log"])
payoff_df = pd.DataFrame.from_dict(payoffs, orient="index", columns=["Payoff"])
payoff_df

In [ ]:
try:
    try:
        ## Step 3 — Iterative auction and convergence analysis
        
        We run multiple auction rounds to observe convergence toward Nash equilibrium. Agents adjust their strategies based on previous outcomes, and we track the total system payoff over time.
    except Exception as e:
        print(f'Error in cell: {e}')
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: invalid syntax (<string>, line 4)...]

def adjust_agent_strategies(
    vessels: List[VesselAgent],
    cranes: List[CraneAgent],
    yards: List[YardBlockAgent],
    allocation: Dict[str, Dict[str, str]],
    learning_rate: float = 0.1,
) -> None:
    """Adjust agent strategies based on previous allocation (simplified learning)."""
    # Cranes: if overloaded, increase energy cost (bid higher) next round
    for crane in cranes:
        tasks_assigned = [t for t, cid in allocation["crane"].items() if cid == crane.id]
        teu_assigned = 0
        for task_id in tasks_assigned:
            for vessel in vessels:
                for task in vessel.tasks:
                    if task.id == task_id:
                        teu_assigned += task.teu
                        break
        if teu_assigned > 2000:  # overloaded threshold
            crane.energy_cost_per_hour *= (1 + learning_rate)
        else:
            crane.energy_cost_per_hour *= (1 - learning_rate * 0.5)  # reduce slightly if underloaded
    # Yards: if highly utilized, increase cost
    for yard in yards:
        tasks_assigned = [t for t, yid in allocation["yard"].items() if yid == yard.id]
        teu_assigned = 0
        for task_id in tasks_assigned:
            for vessel in vessels:
                for task in vessel.tasks:
                    if task.id == task_id:
                        teu_assigned += task.teu
                        break
        utilization = teu_assigned / yard.capacity
        if utilization > 0.8:
            # Increase cost (already reflected in bid function)
            pass
        else:
            # Decrease cost by increasing capacity (simulated)
            yard.transport_capacity *= (1 + learning_rate * 0.2)


def run_iterative_auction(
    vessels: List[VesselAgent],
    cranes: List[CraneAgent],
    yards: List[YardBlockAgent],
    control: TerminalControlAgent,
    rounds: int = 5,
    seed: int = 42,
) -> Dict[str, Any]:
    """Run multiple auction rounds to converge toward equilibrium."""
    history = []
    for r in range(rounds):
        result = run_auction(vessels, cranes, yards, control, log=False)
        payoffs = compute_payoffs(vessels, cranes, yards, result["allocation"], result["auction_log"])
        history.append(
            {
                "round": r + 1,
                "payoffs": payoffs.copy(),
                "allocation": result["allocation"],
                "total_system_payoff": sum(payoffs.values()),
            }
        )
        # Adjust strategies for next round
        adjust_agent_strategies(vessels, cranes, yards, result["allocation"])
    return {"history": history}


iterative_result = run_iterative_auction(vessels, cranes, yards, control, rounds=5, seed=42)
# Plot total system payoff over rounds
rounds = [entry["round"] for entry in iterative_result["history"]]
total_payoffs = [entry["total_system_payoff"] for entry in iterative_result["history"]]
plt.figure(figsize=(6, 3.2))
plt.plot(rounds, total_payoffs, marker="o", color="#2E90FA")
plt.xlabel("Auction Round")
plt.ylabel("Total System Payoff")
plt.title("Convergence toward Nash Equilibrium (Multi-Agent System)")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
try:
    def adjust_agent_strategies(
        vessels: List[VesselAgent],
        cranes: List[CraneAgent],
        yards: List[YardBlockAgent],
        allocation: Dict[str, Dict[str, str]],
        learning_rate: float = 0.1,
    ) -> None:
        """Adjust agent strategies based on previous allocation (simplified learning)."""
        # Cranes: if overloaded, increase energy cost (bid higher) next round
        for crane in cranes:
            tasks_assigned = [t for t, cid in allocation["crane"].items() if cid == crane.id]
            teu_assigned = sum(next(t.teu for t in vessels[0].tasks if t.id == task_id) for task_id in tasks_assigned)
            if teu_assigned > 2000:  # overloaded threshold
                crane.energy_cost_per_hour *= (1 + learning_rate)
            else:
                crane.energy_cost_per_hour *= (1 - learning_rate * 0.5)  # reduce slightly if underloaded
        # Yards: if highly utilized, increase cost
        for yard in yards:
            tasks_assigned = [t for t, yid in allocation["yard"].items() if yid == yard.id]
            teu_assigned = sum(next(t.teu for t in vessels[0].tasks if t.id == task_id) for task_id in tasks_assigned)
            utilization = teu_assigned / yard.capacity
            if utilization > 0.8:
                # Increase cost (already reflected in bid function)
                pass
            else:
                # Decrease cost by increasing capacity (simulated)
                yard.transport_capacity *= (1 + learning_rate * 0.2)
    
    
    def run_iterative_auction(
        vessels: List[VesselAgent],
        cranes: List[CraneAgent],
        yards: List[YardBlockAgent],
        control: TerminalControlAgent,
        rounds: int = 5,
        seed: int = 42,
    ) -> Dict[str, Any]:
        """Run multiple auction rounds to converge toward equilibrium."""
        history = []
        for r in range(rounds):
            result = run_auction(vessels, cranes, yards, control, log=False)
            payoffs = compute_payoffs(vessels, cranes, yards, result["allocation"], result["auction_log"])
            history.append(
                {
                    "round": r + 1,
                    "payoffs": payoffs.copy(),
                    "allocation": result["allocation"],
                    "total_system_payoff": sum(payoffs.values()),
                }
            )
            # Adjust strategies for next round
            adjust_agent_strategies(vessels, cranes, yards, result["allocation"])
        return {"history": history}
    
    
    iterative_result = run_iterative_auction(vessels, cranes, yards, control, rounds=5, seed=42)
    # Plot total system payoff over rounds
    rounds = [entry["round"] for entry in iterative_result["history"]]
    total_payoffs = [entry["total_system_payoff"] for entry in iterative_result["history"]]
    plt.figure(figsize=(6, 3.2))
    plt.plot(rounds, total_payoffs, marker="o", color="#2E90FA")
    plt.xlabel("Auction Round")
    plt.ylabel("Total System Payoff")
    plt.title("Convergence toward Nash Equilibrium (Multi-Agent System)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'run_auction' is not defined


## Step 4 — Disruption scenario: crane failure

We simulate a crane failure after the first round and observe how the multi-agent system redistributes tasks automatically.

In [ ]:
try:
    # Reset agents
    vessels_dis, cranes_dis, yards_dis, control_dis = create_scenario(seed=42)
    # Run one round
    result_round1 = run_auction(vessels_dis, cranes_dis, yards_dis, control_dis, log=False)
    # Simulate crane failure: remove crane C3
    failed_crane_id = "C3"
    cranes_dis = [c for c in cranes_dis if c.id != failed_crane_id]
    print(f"Simulated failure of crane {failed_crane_id}. Remaining cranes: {len(cranes_dis)}")
    # Run second round with remaining cranes
    result_round2 = run_auction(vessels_dis, cranes_dis, yards_dis, control_dis, log=True)
    # Compare payoffs before and after failure
    payoffs_before = compute_payoffs(vessels_dis, cranes_dis + [next(c for c in cranes if c.id == failed_crane_id)], yards_dis, result_round1["allocation"], result_round1["auction_log"])
    payoffs_after = compute_payoffs(vessels_dis, cranes_dis, yards_dis, result_round2["allocation"], result_round2["auction_log"])
    comparison_df = pd.DataFrame({
        "Before Failure": payoffs_before,
        "After Failure": payoffs_after,
    })
    comparison_df["Change"] = comparison_df["After Failure"] - comparison_df["Before Failure"]
    comparison_df.round(2)
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'run_auction' is not defined


## Step 5 — Performance comparison: Multi-agent vs Centralized

We compare the total system payoff of the multi-agent system against a simple centralized baseline (assign tasks greedily to nearest crane).

In [ ]:
try:
    def centralized_baseline(
        vessels: List[VesselAgent],
        cranes: List[CraneAgent],
        yards: List[YardBlockAgent],
    ) -> Dict[str, Any]:
        """Simple centralized baseline: assign each task to nearest crane (ignoring capacity)."""
        allocation = {"crane": {}, "yard": {}}
        for vessel in vessels:
            for task in vessel.tasks:
                # Assign to nearest crane
                nearest_crane = min(cranes, key=lambda c: abs(c.position - int(task.id.split('-')[-1])))
                allocation["crane"][task.id] = nearest_crane.id
                # Assign to first yard with space
                for yard in yards:
                    if yard.capacity - yard.current_utilization >= task.teu:
                        allocation["yard"][task.id] = yard.id
                        yard.current_utilization += task.teu
                        break
        # Compute payoffs (reuse function)
        dummy_log = pd.DataFrame([{"task_id": tid, "crane_cost": 0, "yard_cost": 0} for tid in allocation["crane"]])
        payoffs = compute_payoffs(vessels, cranes, yards, allocation, dummy_log)
        return {"allocation": allocation, "payoffs": payoffs, "total_payoff": sum(payoffs.values())}
    
    
    # Reset agents
    vessels_cmp, cranes_cmp, yards_cmp, control_cmp = create_scenario(seed=42)
    # Multi-agent result (final round)
    ma_result = iterative_result["history"][-1]
    ma_total = ma_result["total_system_payoff"]
    # Centralized baseline
    central_result = centralized_baseline(vessels_cmp, cranes_cmp, yards_cmp)
    central_total = central_result["total_payoff"]
    print(f"Multi-agent total payoff: {ma_total:.2f}")
    print(f"Centralized baseline total payoff: {central_total:.2f}")
    print(f"Improvement: {((ma_total - central_total) / central_total * 100):.1f}%")
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'iterative_result' is not defined


## Step 6 — Why this Tier exists vs Tier 5

### Advantages vs Tier 5
- **Decentralized resilience**: The system automatically reassigns tasks when a crane fails, without a central replanner.
- **Scalability**: Adding more agents (cranes, vessels, yards) does not require redesigning a central optimizer.
- **Local autonomy**: Agents optimize their own objectives while respecting global constraints via auctions.
- **Emergent coordination**: System-wide efficiency emerges from local negotiations, often approaching Nash equilibrium.

### Disadvantages vs Tier 5
- **No global optimum guarantee**: The Nash equilibrium may be suboptimal compared to a centralized optimum.
- **Communication overhead**: Auctions require message passing between agents; in real systems this adds latency.
- **Strategic behavior**: Agents might manipulate bids if they have private information; mechanism design is needed.

### When to use this Tier
- When the terminal has **many heterogeneous resources** that need to coordinate in real time.
- When **resilience to failures** is critical (automatic redistribution without human intervention).
- When you want a **scalable architecture** that can easily add new equipment or vessels.

### How this Tier connects to higher Tiers
- **Tier 9 (Quantum)** could provide quantum algorithms for solving the underlying auction game faster, enabling real-time recomputation of equilibrium.

For now, Tier 6 gives you a **self-organizing terminal ecosystem** where local intelligence leads to global efficiency, with built-in resilience and scalability.